# Vortex Geometry — Entanglement Growth in Random Circuits

Tracks how three collective entanglement order parameters evolve with circuit depth
across random brickwork circuits on a $3 \times 4 = 12$-qubit grid:

| Symbol | Name | Source |
|--------|------|--------|
| $\bar{S}$ | Mean bond entropy | `entropy_map` per-bond average |
| $f_Q$ | QFI density | Tóth–Gühne 2012 multipartite entanglement witness |
| $k$ | Entanglement depth | $k = \lfloor f_Q \rfloor$ certifies genuine $(k+1)$-partite entanglement |

**Prediction:** As circuit depth $d$ grows, both $\bar{S}$ and $f_Q$ grow monotonically
until the circuit saturates the **Tsirelson–Page volume law** $\bar{S} \to \log_2 2 = 1$.

The KLT phase engine detects the chaos regime transition (Z1 → Z3 → Z5) as $f_Q$ grows.

**Protocol:**  
Run random brickwork circuits (Sycamore-style) at depths $d \in \{2, 4, 8, 16\}$  
with `mode="klt_phase"` and `return_entropy_map=True`. Plot the entanglement landscape.

**Change your API key in the cell below, then run all cells.**

In [ ]:
# ── Set your API key ───────────────────────────────────────────────────────────
# Get a free key: curl -s -X POST https://api.qumulator.com/keys \
#   -H 'Content-Type: application/json' -d '{"name": "my-key"}' | python -m json.tool

import os
API_KEY = os.environ.get('QUMULATOR_API_KEY', 'YOUR_KEY_HERE')
API_URL = 'https://api.qumulator.com'

In [ ]:
%pip install qumulator-sdk --quiet
from qumulator import QumulatorClient
import numpy as np
import math
import time

client = QumulatorClient(api_url=API_URL, api_key=API_KEY)
print('SDK ready.')

## Build random brickwork circuits

We use a Sycamore-inspired brickwork layout on a $3 \times 4$ grid.
Each layer applies random $SU(2)$ single-qubit gates followed by a
fixed 2-qubit entangling layer (A, B, C, D alternating column/row pairs).

All randomness is seeded so results are fully reproducible.

In [ ]:
N      = 12
DEPTHS = [2, 4, 8, 16]
SEED   = 42

import random as _random

def build_circuit(n_qubits: int, depth: int, seed: int) -> list:
    """Alternating CX brickwork with random Rz single-qubit rotations."""
    rng   = _random.Random(seed)
    gates = [{"gate": "h", "qubits": [i]} for i in range(n_qubits)]
    for d in range(depth):
        start = d % 2
        for i in range(start, n_qubits - 1, 2):
            gates.append({"gate": "cx", "qubits": [i, i + 1]})
        for i in range(n_qubits):
            theta = rng.uniform(0, 3.14159)
            gates.append({"gate": "rz", "qubits": [i], "params": [theta]})
    return gates

print(f'Qubits : {N}')
print(f'Depths : {DEPTHS}')
print(f'Seed   : {SEED}')
for d in DEPTHS:
    g = build_circuit(N, d, SEED)
    print(f'  depth={d:2d}  gates={len(g)}')

## Run circuits at each depth — collect entropy diagnostics

We submit one circuit per depth level and collect:
- `entropy_map` — per-bond entanglement entropy $S_i$
- `f_Q_density` — QFI density (Tóth–Gühne multipartite witness)
- `entanglement_depth` — certified genuine entanglement depth $k = \lfloor f_Q \rfloor$
- `phase_label` — KLT chaos regime (Z1 = low entropy, Z5 = Haar-random)

> **Note:** The API has a rate limit of 1 request/minute on free keys.
> This cell submits one request per depth level with automatic waits between calls.

In [ ]:
results = []

for idx, depth in enumerate(DEPTHS):
    if idx > 0:
        print(f'  [rate-limit pause 62s]')
        time.sleep(62)

    gates = build_circuit(N, depth, SEED)
    t0 = time.time()
    r = client.circuit.run(
        n_qubits=N,
        gates=gates,
        shots=512,
        mode="klt_phase",
        return_entropy_map=True,
    )
    elapsed = time.time() - t0

    em   = r.entropy_map or []
    s_mean = float(np.mean(em)) if em else 0.0

    results.append({
        "depth":              depth,
        "mean_entropy":       round(s_mean, 4),
        "f_Q_density":        r.f_Q_density,
        "entanglement_depth": r.entanglement_depth,
        "predicted_tvd":      r.predicted_tvd,
        "phase_label":        getattr(r, 'phase_label', None),
        "elapsed":            round(elapsed, 2),
    })

    print(
        f'depth={depth:2d}  S̄={s_mean:.4f}  '
        f'f_Q={r.f_Q_density}  k={r.entanglement_depth}  '
        f'phase={getattr(r, "phase_label", "—")}  '
        f'TVD={r.predicted_tvd}  [{elapsed:.1f}s]'
    )

print('\nDone.')

## Entanglement landscape summary

Print the depth → order-parameter table and verify the volume-law growth.

In [ ]:
S_page = math.log2(2)   # Page value = 1 bit for N/2 partition

print('='*70)
print(' VORTEX GEOMETRY — Entanglement Growth in Random Circuits')
print('='*70)
print(f'{"depth":>6}  {"S̄ (bits)":>10}  {"f_Q":>8}  {"k (depth)":>10}  {"phase":>7}  {"TVD":>7}')
print('-'*70)
for row in results:
    print(
        f'{row["depth"]:>6}  '
        f'{row["mean_entropy"]:>10.4f}  '
        f'{str(row["f_Q_density"]):>8}  '
        f'{str(row["entanglement_depth"]):>10}  '
        f'{str(row["phase_label"]):>7}  '
        f'{str(row["predicted_tvd"]):>7}'
    )
print('='*70)
print(f'Page entropy (volume law saturation): {S_page:.3f} bits')

# Verify monotone growth in mean entropy
entropies = [row['mean_entropy'] for row in results]
growing   = all(entropies[i] <= entropies[i+1] + 0.05 for i in range(len(entropies)-1))
print(f'\nEntropy grows with depth: {"✓ YES" if growing else "~ non-monotone (expected at low depth)"}')

# Verify deepest circuit is more entangled than shallowest
assert entropies[-1] > entropies[0], 'Deepest circuit should be more entangled than shallowest'
print(f'Deep > shallow entanglement: ✓  ({entropies[0]:.4f} → {entropies[-1]:.4f} bits)')